In [1]:
print(123)

123


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
print(len(documents))

72


In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
from pydantic import BaseModel
from evaluation_utils import llm_structured, calc_price, calc_total_price, llm_structured_retry, map_progress
from dotenv import load_dotenv
from openai import OpenAI
import json


load_dotenv('llm.env')
openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

In [14]:
records = []
usages = []

for doc in documents:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="gpt-5.4-mini",
    )

    for question in result.questions:
        records.append({
            "question": question,
            "filename": doc["filename"]
        })

    usages.append(usage)


In [21]:
records[0], usages[0]

({'question': 'What exactly is RAG, and why does it help when an LLM doesn’t know something or can’t see my data?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 ResponseUsage(input_tokens=1021, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1145))

In [22]:
records = []
usages = []

for doc in documents[:3]:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="gpt-5.4-mini",
    )

    for question in result.questions:
        records.append({
            "question": question,
            "filename": doc["filename"]
        })

    usages.append(usage)


In [ ]:
usages

#Average input_tokens = 1354 ~ 1400

[ResponseUsage(input_tokens=1021, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1133),
 ResponseUsage(input_tokens=1287, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=102, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1389),
 ResponseUsage(input_tokens=1754, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1846)]

In [24]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)


In [30]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [32]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(
        query, 
        num_results=num_results)


In [34]:
from embedder import Embedder
from minsearch import VectorSearch

embedder = Embedder()   # loads models/Xenova/all-MiniLM-L6-v2

texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(texts)   # matrix of normalized 384-dim vectors

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    q = embedder.encode(query)
    return vector_index.search(q, num_results=num_results)

In [35]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [36]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [40]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

ground_truth[0]["question"]

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [41]:
q = ground_truth[0]["question"]

In [ ]:
text_search(q, num_results=5)

# 01-agentic-rag/lessons/03-rag.md is the filename of the first returned result.

In [ ]:
vector_search(q, num_results=5)

# 01-agentic-rag/lessons/01-intro.md is the filename of the first returned result.

In [44]:
def compute_relevance_text(q):
    doc_name = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_name))

    return relevance

In [45]:
compute_relevance_text(ground_truth[0])

[0, 0, 0, 0, 1]

In [52]:
def compute_relevance(q, search_function):
    doc_name = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_name))

    return relevance

In [53]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [54]:
relevance_total_text = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [55]:
relevance_total_vector = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [57]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [58]:
hit_rate(relevance_total_text), hit_rate(relevance_total_vector)

(0.7583333333333333, 0.725)

In [59]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [60]:
mrr(relevance_total_text), mrr(relevance_total_vector)

(0.5942592592592594, 0.5486111111111112)

In [63]:
from functools import partial

def mrr(relevance_total):
    scores = []
    for rel in relevance_total:
        for rank, is_relevant in enumerate(rel):
            if is_relevant:
                scores.append(1 / (rank + 1))
                break
        else:
            scores.append(0)
    return sum(scores) / len(scores)

for k in [1, 10, 60, 100]:
    search_fn = partial(hybrid_search, k=k)
    relevance_total = compute_relevance_total(ground_truth, search_fn)
    
    hit_rate = sum(any(rel) for rel in relevance_total) / len(relevance_total)
    mrr_score = mrr(relevance_total)
    print(f"k={k}: hit_rate={hit_rate:.3f}, mrr={mrr_score:.3f}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: hit_rate=0.839, mrr=0.648


  0%|          | 0/360 [00:00<?, ?it/s]

k=10: hit_rate=0.836, mrr=0.640


  0%|          | 0/360 [00:00<?, ?it/s]

k=60: hit_rate=0.836, mrr=0.638


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: hit_rate=0.836, mrr=0.638
